# ManuAI

## Download the Data

This script will download the data required to train ManuAI.

It sources from:
- Xeno-Canto: [https://www.xeno-canto.org/](https://www.xeno-canto.org/)
- Kaggle: [https://www.kaggle.com/datasets/ollypowell/new-zealand-bird-sound](https://www.kaggle.com/datasets/ollypowell/new-zealand-bird-sound)

## Imports

In [1]:
import os, re, requests, subprocess, yaml, concurrent.futures
import pandas as pd
import tqdm
import datetime
from dotenv import load_dotenv
import kagglehub
from concurrent.futures import ThreadPoolExecutor, as_completed
import soundfile as sf  # or use librosa.get_duration
from pathlib import Path
import shutil

load_dotenv()

XENO_API_KEY = os.getenv('XC_API_KEY')
RECORDING_TYPES = ['song', 'call', 'territorial call', 'alarm call']
KAGGLE_SOURCES = ['DOC_001_Tier1', 'DOC_002_DuncanBayParinga']
redacted_title = ['south_island_', 'north_island_', 'new_zealand_', 'australian_']
#KAGGLE_SOURCES = ['DOC_001_Tier1', 'DOC_002_DuncanBayParinga', 'DOC_003_XenoCanto'] # For testing --- IGNORE ---
MAX_WORKERS = os.cpu_count() - 1 or 1
SR = 44100

with open('types.yaml', 'r') as f:
    BIRD_SPECIES = yaml.safe_load(f)

if not XENO_API_KEY:
    raise ValueError('Set XC_API_KEY in .env file.')

In [2]:
def sanitize(text):
    return re.sub(r'[^a-z0-9_]', '', str(text).lower().replace(' ', '_'))

def parse_length(length_str):
    try:
        dt = datetime.datetime.strptime(length_str, '%M:%S')
        return dt.minute * 60 + dt.second
    except:
        return 0

def to_wav(src, dest):
    subprocess.run(
        ['ffmpeg', '-y', '-i', src, '-ar', str(SR), '-ac', '1', dest],
        check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

def resolve_label(scientific: str) -> str:
    """Return the types.yaml label (e.g. 'Tui', 'GreyWarbler') for a scientific name, or '' if unknown."""
    normalized = scientific.lower().replace('_', ' ').strip()
    for label, sci_names in BIRD_SPECIES.items():
        if normalized in [s.lower() for s in sci_names]:
            return label
    return ''

## Gather location to store data

In [3]:
scientific_to_english = {
    sci.lower(): eng.lower()
    for eng, sciences in BIRD_SPECIES.items()
    for sci in sciences
}

# --- Interactive Directory Browser ---
def get_dir_info(path: Path) -> str:
    """Get free space for a directory."""
    try:
        stat = os.statvfs(path)
        free_gb = (stat.f_bavail * stat.f_frsize) / (1024**3)
        return f"({free_gb:.1f} GB free)"
    except:
        return ""

def list_subdirs(path: Path) -> list[Path]:
    """List subdirectories, excluding hidden ones."""
    try:
        return sorted([
            d for d in path.iterdir() 
            if d.is_dir() and not d.name.startswith('.')
        ], key=lambda x: x.name.lower())
    except PermissionError:
        return []

def browse_directory(start_path: Path = None) -> str | None:
    """Interactive directory browser. Returns selected path or None."""
    
    # Start at /Volumes or provided path
    if start_path is None:
        current = Path('/Volumes')
    else:
        current = Path(start_path)
    
    while True:
        print(f"\n{'='*50}")
        print(f"📁 {current}  {get_dir_info(current)}")
        print('='*50)
        
        subdirs = list_subdirs(current)
        
        # Show navigation options
        print("\n[..] Go up")
        print("[s]  SELECT this folder")
        print("[n]  Create NEW folder here")
        print("[q]  Quit/Cancel")
        print()
        
        # Show subdirectories
        if subdirs:
            for i, d in enumerate(subdirs, 1):
                print(f"[{i}]  {d.name}/")
        else:
            print("(no subdirectories)")
        
        choice = input("\nEnter choice: ").strip().lower()
        
        if choice == 'q':
            return None
        elif choice == 's':
            return str(current)
        elif choice == '..':
            if current.parent != current:  # Not at root
                current = current.parent
        elif choice == 'n':
            name = input("New folder name: ").strip()
            if name and not name.startswith('.'):
                new_dir = current / name
                try:
                    new_dir.mkdir(parents=True, exist_ok=True)
                    print(f"Created: {new_dir}")
                    current = new_dir
                except Exception as e:
                    print(f"Error creating folder: {e}")
        elif choice.isdigit():
            idx = int(choice)
            if 1 <= idx <= len(subdirs):
                current = subdirs[idx - 1]
        # Direct path entry
        elif choice.startswith('/') or choice.startswith('~'):
            test_path = Path(choice).expanduser()
            if test_path.exists() and test_path.is_dir():
                current = test_path
            else:
                print(f"Path not found: {choice}")

def select_data_directory():
    """Main directory selection with config fallback."""
    # Check for existing config
    default_data_dir = ''
    config_path = Path('ManuAI.config')
    if config_path.exists():
        try:
            with open(config_path, 'r') as f:
                cfg = yaml.safe_load(f) or {}
            default_data_dir = str(cfg.get('data_directory', '')).strip()
        except:
            pass
    
    print("\n=== ManuAI Data Directory Setup ===")
    
    if default_data_dir and Path(default_data_dir).exists():
        print(f"\nPrevious: {default_data_dir}")
        use_prev = input("Use previous location? [Y/n]: ").strip().lower()
        if use_prev != 'n':
            return default_data_dir
    
    print("\nStarting directory browser at /Volumes...")
    return browse_directory()

# Run selection
data_dir = select_data_directory()

if not data_dir:
    raise SystemExit('No directory selected.')

data_dir = str(Path(data_dir).expanduser().resolve())
Path(data_dir).mkdir(parents=True, exist_ok=True)
kaggle_dir = Path('archive')
xeno_dir = Path(data_dir) / 'xeno_canto'
save_dir = Path(data_dir) / 'processed'

print(f'\n✓ Data will be stored in: {data_dir}')

with open('ManuAI.config', 'w') as f:
    yaml.dump({'data_directory': data_dir}, f)


=== ManuAI Data Directory Setup ===

Previous: /Volumes/HAZZA/MANUAI/DATA

✓ Data will be stored in: /Volumes/HAZZA/MANUAI/DATA


In [4]:
VALID_SPECIES = {
    sanitize(sci_name)
    for names in BIRD_SPECIES.values()
    for sci_name in names
}
print(f'Target species (sanitized): {len(VALID_SPECIES)} names')


Target species (sanitized): 64 names


## Xeno-Canto

In [5]:
def fetch_xeno(recording_types):
    recordings = []
    for rec_type in recording_types:
        page = 1
        query = f'cnt:"New Zealand" grp:"birds" type:"{rec_type}" q:">C"'
        print(f'Fetching {rec_type}...')
        while True:
            data = requests.get(
                'https://xeno-canto.org/api/3/recordings',
                params={'query': query, 'key': XENO_API_KEY, 'page': page}
            ).json()
            if page == 1:
                print(f'{data["numRecordings"]} recordings, {data["numSpecies"]} species')
            recordings.extend(data.get('recordings', []))
            if page >= int(data.get('numPages', 1)):
                break
            page += 1

    recordings = list({r['id']: r for r in recordings if 'id' in r}.values())
    print(f'Total viable recordings fetched: {len(recordings)}')

    return [
        rec for rec in recordings
        if 'gen' in rec and 'sp' in rec
        and sanitize(f"{rec['gen']} {rec['sp']}") in VALID_SPECIES
    ]

In [6]:
xeno_recordings = fetch_xeno(RECORDING_TYPES)
print(f'Filtered to {len(xeno_recordings)} recordings')

def _strip_redacted(name: str) -> str:
    for prefix in redacted_title:
        name = name.replace(prefix, '')
    return name.strip('_')

def build_xc_row(r):
    genus = sanitize(r.get('gen', ''))
    species_epithet = sanitize(r.get('sp', ''))
    scientific = f"{genus}_{species_epithet}"

    english = r.get('en', 'Unknown')
    for prefix in ['North Island ', 'South Island ', 'New Zealand ', 'Australian ']:
        english = english.replace(prefix, '')

    file_id = f"xc{r['id']}"

    return {
        'id': file_id,
        'filename': f'pre/{file_id}.wav',
        'source': 'xeno-canto',
        'label': resolve_label(scientific),
        'scientific_name': scientific,
        'genus': genus,
        'species_epithet': species_epithet,
        'english_name': english,
        'ebird_code': '',
        'recording_type': r.get('type', ''),
        'quality': r.get('q', ''),
        'length_seconds': parse_length(r.get('length', '0:00')),
        'sample_rate': int(r.get('smp', SR)),
        'license': r.get('lic', ''),
        'location': r.get('loc', ''),
        'country': r.get('cnt', ''),
        'lat': r.get('lat', ''),
        'lon': r.get('lon', ''),
        'date': r.get('date', ''),
        'sex': r.get('sex', ''),
        'also': ';'.join(r.get('also', [])) if isinstance(r.get('also'), list) else '',
        'remarks': r.get('rmk', ''),
    }

xc_rows = [build_xc_row(r) for r in xeno_recordings if 'file' in r]
print(f'Built {len(xc_rows)} Xeno-Canto metadata rows')

Fetching song...
906 recordings, 88 species
Fetching call...
1028 recordings, 142 species
Fetching territorial call...
0 recordings, 0 species
Fetching alarm call...
61 recordings, 29 species
Total viable recordings fetched: 1728
Filtered to 807 recordings
Built 807 Xeno-Canto metadata rows


In [7]:
# --- Optimized Xeno-Canto download for USB ---
# Single-threaded + connection pooling + large buffers + skip existing

import tempfile

dest_root = Path(data_dir) / 'pre'
dest_root.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 4 * 1024 * 1024  # 4MB chunks

def download_single_xeno(rec, session, dest_root):
    """Download one recording, convert to wav, save to dest_root."""
    if 'file' not in rec:
        return None
    
    file_id = f"xc{rec['id']}"
    dest_path = dest_root / f"{file_id}.wav"
    
    # Skip if exists
    if dest_path.exists() and dest_path.stat().st_size > 0:
        return str(dest_path)
    
    file_url = rec['file']
    if file_url.startswith('//'):
        file_url = 'https:' + file_url
    
    try:
        ext = os.path.splitext(file_url)[1].lower() or '.mp3'
        
        # Download to temp file first
        with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as tmp:
            tmp_path = tmp.name
            with session.get(file_url, stream=True, timeout=60) as r:
                if r.status_code != 200:
                    return None
                for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                    if chunk:
                        tmp.write(chunk)
        
        # Convert to wav if needed
        if ext != '.wav':
            to_wav(tmp_path, str(dest_path))
            Path(tmp_path).unlink(missing_ok=True)
        else:
            # Copy with large buffer to USB
            with open(tmp_path, 'rb') as src, open(dest_path, 'wb') as dst:
                while chunk := src.read(CHUNK_SIZE):
                    dst.write(chunk)
            Path(tmp_path).unlink(missing_ok=True)
        
        return str(dest_path)
    except Exception as e:
        print(f'Error {file_id}: {e}')
        return None

# Single-threaded download with session (connection pooling)
session = requests.Session()
adapter = requests.adapters.HTTPAdapter(pool_connections=10, pool_maxsize=10)
session.mount('https://', adapter)
session.mount('http://', adapter)

xeno_paths = []
downloaded, skipped = 0, 0

for rec in tqdm.tqdm(xeno_recordings, desc='Downloading Xeno-Canto'):
    result = download_single_xeno(rec, session, dest_root)
    if result:
        if Path(result).stat().st_size > 0:
            xeno_paths.append(result)
            # Check if it was already there
            downloaded += 1
        else:
            skipped += 1

session.close()
print(f'Xeno-Canto: {len(xeno_paths)} files ready')

Xeno-Canto: 807 files ready


## Kaggle

Pre-download Kaggle dataset manually, then run this cell to move it to the correct location.
Expects dataset to be downloaded to `./archive/`.

## Process Kaggle

In [8]:
archive_root = Path(kaggle_dir)
dest_root = Path(data_dir) / 'pre'
dest_root.mkdir(parents=True, exist_ok=True)

if not archive_root.exists():
    raise FileNotFoundError(f'Kaggle archive not found: {archive_root}')

# Build eBird -> species metadata mapping from naming maps.
ebird_map = {}
for source in KAGGLE_SOURCES:
    map_path = archive_root / source / source / 'bird_naming_map.csv'
    if not map_path.exists():
        continue
    for _, row in pd.read_csv(map_path).iterrows():
        code = str(row.get('eBird', '')).strip()
        if code and code not in ebird_map:
            ebird_map[code] = row

# Build set of valid eBird codes (those that map to species in types.yaml)
valid_ebird_codes = set()
for code, row in ebird_map.items():
    sci_name = str(row.get('ScientificName', '')).strip().lower()
    if sanitize(sci_name) in VALID_SPECIES:
        valid_ebird_codes.add(code)

print(f'Valid eBird codes from types.yaml: {len(valid_ebird_codes)} / {len(ebird_map)}')

audio_exts = {'.flac', '.wav', '.mp3', '.ogg', '.m4a'}
all_files = [p for p in archive_root.rglob('*') if p.is_file() and p.suffix.lower() in audio_exts]

# FILTER: Only files in valid species folders
files = [f for f in all_files if f.parent.name in valid_ebird_codes]
print(f'Filtered: {len(files)} / {len(all_files)} audio files match types.yaml species')

# --- Optimized for USB: single-threaded with large buffer + skip existing ---
BUFFER_SIZE = 4 * 1024 * 1024  # 4MB buffer — reduces syscalls on USB

def fast_copy(src: Path, dest: Path) -> bool:
    """Copy with large buffer, skip if exists with same size."""
    if dest.exists() and dest.stat().st_size == src.stat().st_size:
        return False  # Already copied
    with open(src, 'rb') as fsrc, open(dest, 'wb') as fdst:
        while chunk := fsrc.read(BUFFER_SIZE):
            fdst.write(chunk)
    return True

def build_row(idx: int, src: Path) -> dict:
    """Build metadata row for a file."""
    ebird = src.parent.name
    species_row = ebird_map.get(ebird)
    ext = src.suffix.lower()
    filename = f'kg_{idx:07d}{ext}'
    
    if species_row is not None:
        common = str(species_row.get('CommonName', species_row.get('ExtraName', ebird))).strip()
        for prefix in ['North Island ', 'South Island ', 'New Zealand ', 'Australian ']:
            common = common.replace(prefix, '')
        sci_parts = str(species_row.get('ScientificName', '')).strip().lower().split()
        genus = sci_parts[0] if len(sci_parts) > 0 else 'unknown'
        species_ep = sci_parts[1] if len(sci_parts) > 1 else 'sp'
        scientific = f"{genus} {species_ep}"
    else:
        common, scientific, genus, species_ep = ebird, '', 'unknown', 'sp'
    
    return {
        'id': f'kg_{idx:07d}',
        'filename': f'pre/{filename}',
        'source': 'kaggle',
        'label': resolve_label(scientific),
        'scientific_name': scientific,
        'genus': genus,
        'species_epithet': species_ep,
        'english_name': common,
        'ebird_code': ebird,
        'recording_type': '', 'quality': '', 'length_seconds': '',
        'sample_rate': '', 'license': '', 'location': '',
        'country': 'New Zealand', 'lat': '', 'lon': '',
        'date': '', 'sex': '', 'also': '', 'remarks': '',
    }

# Single-threaded copy — USB drives are faster without parallel I/O
kg_rows = []
copied, skipped = 0, 0

for idx, src in tqdm.tqdm(enumerate(files, 1), total=len(files), desc='Copying Kaggle'):
    ext = src.suffix.lower()
    filename = f'kg_{idx:07d}{ext}'
    dest = dest_root / filename
    
    if fast_copy(src, dest):
        copied += 1
    else:
        skipped += 1
    
    kg_rows.append(build_row(idx, src))

print(f'Done: {copied} copied, {skipped} skipped (already existed)')

Valid eBird codes from types.yaml: 26 / 138
Filtered: 253747 / 271365 audio files match types.yaml species


Copying Kaggle:  45%|████▍     | 113557/253747 [28:53:49<35:40:27,  1.09it/s]    


KeyboardInterrupt: 

In [ ]:
# Merge Xeno-Canto and Kaggle metadata and write recordings.csv
all_rows = xc_rows + kg_rows
recordings_csv = Path(data_dir) / 'recordings.csv'
pd.DataFrame(all_rows).to_csv(recordings_csv, index=False)
print(f'Wrote {len(all_rows)} rows ({len(xc_rows)} xeno-canto, {len(kg_rows)} kaggle) -> {recordings_csv}')